# E2E Pipeline: Batch Log ScoringProcesses all `.log` files in a folder → outputs per-role, per-turn scores to a folder.**Modes:**- `two-step` (default): ClaimExtractor (LLM) + ClaimScorer (embeddings) → 4 score dimensions- `llm-judge`: GPT-4o evaluates all 14 failure modes per interaction**Output:** `{name}_scored.json` per log + `_summary.json` + `_summary.csv` aggregate**Setup:** Run cells in order. Configure `INPUT_DIR`, `OUTPUT_DIR`, `MODE` in the cell below.

In [ ]:
# ⚙️ Configuration — change these before "Run All"INPUT_DIR = "examples/"            # folder with .log files (flat or nested subdirs)OUTPUT_DIR = "scored_results/"      # output folder for scored JSONs + summary CSVsMODE = "two-step"                   # "two-step" | "llm-judge"LLM_MODEL = "gpt-4o"               # model for claim extraction (two-step) or judge (llm-judge)EMBEDDING_MODEL = "text-embedding-3-small"CACHE_PLAYBOOK = True              # skip rebuilding playbook if _playbook.json existsTAXONOMY_DIR = "CLAUDE_TODO/taxonomy_definitions_examples"import osos.makedirs(OUTPUT_DIR, exist_ok=True)print(f"INPUT_DIR:   {os.path.abspath(INPUT_DIR)}")print(f"OUTPUT_DIR:  {os.path.abspath(OUTPUT_DIR)}")print(f"MODE:        {MODE}")print(f"LLM_MODEL:   {LLM_MODEL}")

In [ ]:
# 📦 Imports & environment setupimport json, sys, time, csv, re, textwrapfrom pathlib import Pathfrom collections import defaultdictfrom typing import List, Dict, Any, Optional, Tupleimport dotenvdotenv.load_dotenv(override=True)from openai import OpenAI# Ensure project root on path_project_root = Path(os.getcwd()).resolve()if str(_project_root) not in sys.path:    sys.path.insert(0, str(_project_root))from chatdev.analyzer.parser import LogParserfrom chatdev.analyzer.convert_to_playbook import api_to_playbookfrom chatdev.analyzer.bug_detector import BugDetector, ScoredPlaybook_client = OpenAI(    api_key=os.environ.get("OPENAI_API_KEY"),    base_url=os.environ.get("BASE_URL"),)print(f"OpenAI client: base_url={_client.base_url}")print("All imports OK ✓")

## Step 1: Log DiscoveryScans `INPUT_DIR` recursively for `.log` files. For each log, checks whether a sibling `api_records.jsonl` exists.

In [ ]:
def discover_log_files(input_dir: str) -> List[Dict[str, Any]]:    """Walk input_dir (flat or nested) and find all .log files with optional api_records.jsonl."""    input_path = Path(input_dir).resolve()    if not input_path.exists():        raise FileNotFoundError(f"Input directory not found: {input_path}")    entries = []    for log_path in sorted(input_path.rglob("*.log")):        api_path = log_path.parent / "api_records.jsonl"        entries.append({            "name": log_path.stem,            "log_path": str(log_path),            "api_records_path": str(api_path) if api_path.exists() else None,            "dir": str(log_path.parent),        })    return entriesentries = discover_log_files(INPUT_DIR)print(f"Discovered {len(entries)} log file(s):")for i, e in enumerate(entries):    has_api = "✓" if e["api_records_path"] else "✗"    size_kb = Path(e["log_path"]).stat().st_size / 1024    print(f"  [{i+1:3d}] {e['name']}  ({size_kb:.0f} KB)  api_records.jsonl: {has_api}")if not entries:    print("⚠ No .log files found. Check INPUT_DIR variable.")

## Step 2: Build PlaybooksEach `.log` is parsed by `LogParser` → structured JSON.- If `api_records.jsonl` exists → `api_to_playbook()` (matches API responses to log events).- Otherwise → `build_playbook_from_parsed_log()` (extracts `agent_message` events directly).A playbook has the structure: `{role: [{turn, phase, phase_turn, prompt, output}, ...]}`.

In [ ]:
def build_playbook_from_parsed_log(parsed_log_path: str) -> Tuple[Dict[str, Any], str]:    # Build playbook dict from LogParser output JSON — no api_records.jsonl needed.    #    # Extracts agent_message events, groups by sender (role), and assigns sequential turns.    # Also extracts the initial task_prompt from the preprocessing event.    #    # Returns:    #     (playbook, task_prompt)    #     playbook: {role: [{turn, phase, phase_turn, prompt, output}, ...]}    #     task_prompt: the original task description string    with open(parsed_log_path, 'r', encoding='utf-8') as f:        log_data = json.load(f)    events = log_data.get('events', [])    agent_messages = [e for e in events if e.get('event_type') == 'agent_message']    if not agent_messages:        print(f"    ⚠ WARNING: No agent_message events found")        return {}, ""    # Extract task_prompt from preprocessing event    task_prompt = ""    for e in events:        if e.get('event_type') == 'preprocessing':            task_prompt = e.get('task_prompt', '')            break    # Group by sender (role), preserving chronological order    playbook = defaultdict(list)    for am in agent_messages:        sender = am.get('sender', 'Unknown')        system_prompt = am.get('system_prompt', '')        message = am.get('message', '')        if not message.strip():            continue        playbook[sender].append({            "turn": len(playbook[sender]) + 1,            "phase": am.get('phase_name', 'Unknown'),            "phase_turn": am.get('turn', 0),            "prompt": system_prompt,            "output": message,        })    return dict(playbook), task_promptdef _is_playbook_truncated(playbook: dict, parsed_json_path: str) -> bool:    # Detect if playbook outputs are truncated vs the parsed JSON source."    # Quick check: first non-empty output length    for role, ints in playbook.items():        if not isinstance(ints, list):            continue        for ix in ints:            out = ix.get('output', '')            if len(out) > 100:                return False  # at least one long output = not truncated            break        break    # All outputs <= 100 chars — compare with parsed JSON    try:        with open(parsed_json_path, 'r', encoding='utf-8') as f:            log_data = json.load(f)        for ev in log_data.get('events', []):            if ev.get('event_type') == 'agent_message':                msg = ev.get('message', '') or ''                if len(msg) > 100:                    return True  # parsed JSON has longer content → playbook truncated                break    except Exception:        pass    return False# ---- Main loop: parse + build playbook for each log ----parser = LogParser()playbook_cache: Dict[str, Dict[str, Any]] = {}for i, entry in enumerate(entries):    log_path = entry["log_path"]    name = entry["name"]    print(f"[{i+1}/{len(entries)}] {name}")    parsed_json_path = Path(entry["dir"]) / f"{name}.json"    # Parse log to JSON (skip if cached)    if not parsed_json_path.exists():        print(f"  Parsing log...")        try:            parser.parse(log_path, str(parsed_json_path))        except Exception as e:            print(f"  ❌ Parse error: {e}")            continue    else:        print(f"  Parsed JSON cached ✓")    playbook_path = Path(entry["dir"]) / f"{name}_playbook.json"    # Build or load playbook    if CACHE_PLAYBOOK and playbook_path.exists():        with open(playbook_path, 'r', encoding='utf-8') as f:            playbook = json.load(f)        if _is_playbook_truncated(playbook, str(parsed_json_path)):            print(f"  ⚠ Cached playbook has truncated content (<=100 chars). Rebuilding...")            playbook, task_prompt = build_playbook_from_parsed_log(str(parsed_json_path))            if not playbook:                print(f"  ❌ Empty playbook, skipping this log")                continue            with open(playbook_path, 'w', encoding='utf-8') as f:                json.dump(playbook, f, ensure_ascii=False, indent=2)        else:            print(f"  Loading cached playbook...")            with open(parsed_json_path, 'r', encoding='utf-8') as f:                log_data = json.load(f)            task_prompt = ""            for ev in log_data.get('events', []):                if ev.get('event_type') == 'preprocessing':                    task_prompt = ev.get('task_prompt', '')                    break    elif entry["api_records_path"]:        print(f"  Building playbook via api_records.jsonl...")        try:            api_to_playbook(entry["api_records_path"], str(parsed_json_path), str(playbook_path))            with open(playbook_path, 'r', encoding='utf-8') as f:                playbook = json.load(f)            with open(parsed_json_path, 'r', encoding='utf-8') as f:                log_data = json.load(f)            task_prompt = ""            for ev in log_data.get('events', []):                if ev.get('event_type') == 'preprocessing':                    task_prompt = ev.get('task_prompt', '')                    break        except Exception as e:            print(f"  ❌ api_to_playbook error: {e}")            continue    else:        print(f"  Building playbook from agent_message events (no api_records.jsonl)...")        playbook, task_prompt = build_playbook_from_parsed_log(str(parsed_json_path))        if not playbook:            print(f"  ❌ Empty playbook, skipping this log")            continue        with open(playbook_path, 'w', encoding='utf-8') as f:            json.dump(playbook, f, ensure_ascii=False, indent=2)    if not isinstance(playbook, dict) or not playbook:        print(f"  ❌ Invalid playbook, skipping")        continue    num_roles = len(playbook)    num_ints = sum(len(v) for v in playbook.values())    tp_snippet = (task_prompt or "")[:80]    print(f"  → {num_roles} roles, {num_ints} interactions  |  task: "{tp_snippet}..."")    playbook_cache[name] = {        "entry": entry,        "playbook": playbook,        "playbook_path": str(playbook_path),        "task_prompt": task_prompt,    }print(f"\n{'='*60}")print(f"Built {len(playbook_cache)}/{len(entries)} playbook(s)")

## Step 3: Score Each Log**Two-step mode** (`BugDetector`):ClaimExtractor (LLM) extracts atomic claims from each output → ClaimScorer computes:- `support_score` (FM-2.2/2.3): groundedness against initial task- `norm_score` (FM-1.1/1.2): compliance with extracted rules- `repetition_score` (FM-1.3): similarity to previous outputs- `plan_action_alignment` (FM-2.6): plan-to-action match**LLM-judge mode**:GPT-4o evaluates each interaction for all 14 failure modes (binary yes/no).

In [ ]:
# ---- LLM Judge helpers ----def _load_taxonomy(taxonomy_dir: str) -> Tuple[str, str, Dict[str, str]]:    """Load failure mode definitions, examples, and name mapping."""    defs_path = Path(taxonomy_dir) / "definitions.txt"    ex_path = Path(taxonomy_dir) / "examples.txt"    definitions = open(str(defs_path), 'r', encoding='utf-8').read() if defs_path.exists() else ""    examples = open(str(ex_path), 'r', encoding='utf-8').read() if ex_path.exists() else ""    fm_names = {        '1.1': 'Disobey Task Specification', '1.2': 'Disobey Role Specification',        '1.3': 'Step Repetition', '1.4': 'Loss of Conversation History',        '1.5': 'Unaware of Termination Conditions', '2.1': 'Conversation Reset',        '2.2': 'Fail to Ask for Clarification', '2.3': 'Task Derailment',        '2.4': 'Information Withholding', '2.5': 'Ignored Other Agent Input',        '2.6': 'Action-Reasoning Mismatch', '3.1': 'Premature Termination',        '3.2': 'Weak Verification', '3.3': 'No or Incorrect Verification',    }    return definitions, examples, fm_namesdef _build_judge_prompt(interaction: dict, task_prompt: str, definitions: str, examples: str) -> str:    """Build the LLM judge evaluation prompt for a single interaction."""    trace = f"TASK: {task_prompt}\n\nPROMPT:\n{interaction.get('prompt', '')[:3000]}\n\nOUTPUT:\n{interaction.get('output', '')[:5000]}"    prompt = (        "Below I will provide a multiagent system trace. Analyze the system behaviour.\n"        "There are several failure modes in multiagent systems I identified. Tell me if you encounter any of them, as a binary yes or no.\n"        "Also tell me whether the task is successfully completed or not, as a binary yes or no.\n"        "Here are the things you should answer. Start after the @@ sign and end before the next @@ sign:\n"        "*** begin of things you should answer *** @@\n"        "A. Freeform text summary of the problems: <summary>\n"        "B. Whether the task is successfully completed: <yes or no>\n"        "C. Whether you encounter any of the failure modes:\n"        "1.1 Disobey Task Specification: <yes or no>\n"        "1.2 Disobey Role Specification: <yes or no>\n"        "1.3 Step Repetition: <yes or no>\n"        "1.4 Loss of Conversation History: <yes or no>\n"        "1.5 Unaware of Termination Conditions: <yes or no>\n"        "2.1 Conversation Reset: <yes or no>\n"        "2.2 Fail to Ask for Clarification: <yes or no>\n"        "2.3 Task Derailment: <yes or no>\n"        "2.4 Information Withholding: <yes or no>\n"        "2.5 Ignored Other Agent Input: <yes or no>\n"        "2.6 Action-Reasoning Mismatch: <yes or no>\n"        "3.1 Premature Termination: <yes or no>\n"        "3.2 Weak Verification: <yes or no>\n"        "3.3 No or Incorrect Verification: <yes or no>\n"        "@@*** end of your answer ***\n"        f"Here is the trace:\n{trace}\n\n"        f"Definitions of the failure modes:\n{definitions}\n\n"        f"Examples of the failure modes:\n{examples}"    )    return promptdef _parse_judge_response(response: str) -> dict:    """Parse LLM judge response into structured dict."""    result = {'summary': '', 'task_completed': None, 'failure_modes': {}}    if not response:        return result    cleaned = response.strip().replace('@@', '')    # Summary    sm = re.search(r'A\.\s*(.*?)(?=\nB\.)', cleaned, re.DOTALL)    if sm:        result['summary'] = sm.group(1).strip()    # Task completed    tm = re.search(r'B\.\s*(yes|no)', cleaned, re.IGNORECASE)    if tm:        result['task_completed'] = tm.group(1).lower() == 'yes'    # Failure modes    for fm_code in ['1.1','1.2','1.3','1.4','1.5','2.1','2.2','2.3','2.4','2.5','2.6','3.1','3.2','3.3']:        pat = rf"{re.escape(fm_code)}\s*[\:\-]?\s*(yes|no)"        m = re.search(pat, cleaned, re.IGNORECASE)        result['failure_modes'][fm_code] = (m.group(1).lower() == 'yes') if m else False    result['raw_response'] = response    return resultdef run_llm_judge(playbook: dict, task_prompt: str, client, model: str,                  taxonomy_dir: str) -> List[dict]:    """Run LLM judge on each interaction in a playbook."""    definitions, examples, fm_names = _load_taxonomy(taxonomy_dir)    results = []    total = sum(len(v) for v in playbook.values()) if isinstance(playbook, dict) else 0    idx = 0    for role, interactions in playbook.items():        if not isinstance(interactions, list):            continue        for ix_data in interactions:            idx += 1            prompt_text = _build_judge_prompt(ix_data, task_prompt, definitions, examples)            raw = None            for attempt in range(3):                try:                    resp = client.chat.completions.create(                        model=model,                        messages=[{"role": "user", "content": prompt_text}],                        temperature=1.0,                    )                    raw = resp.choices[0].message.content if resp.choices else ""                    break                except Exception as e:                    if attempt < 2:                        time.sleep(2 ** attempt)                    else:                        print(f"    ⚠ LLM judge API error after 3 retries: {e}")            parsed = _parse_judge_response(raw)            parsed['role'] = role            parsed['turn'] = ix_data.get('turn', idx)            parsed['phase'] = ix_data.get('phase', 'Unknown')            results.append(parsed)            detected = [k for k, v in parsed.get('failure_modes', {}).items() if v]            print(f"  [{idx}/{total}] {role} turn {ix_data.get('turn', '?')}: "                  f"completed={parsed['task_completed']}, failures={detected or 'none'}")    return results# ---- Scoring dispatcher ----all_results = []definitions, examples, fm_names = _load_taxonomy(TAXONOMY_DIR)  # preload for llm-judgefor i, (name, cache) in enumerate(playbook_cache.items()):    playbook = cache["playbook"]    playbook_path = cache["playbook_path"]    task_prompt = cache["task_prompt"]    entry = cache["entry"]    print(f"\n[{i+1}/{len(playbook_cache)}] Scoring: {name}")    if MODE == "two-step":        print(f"  Mode: two-step (ClaimExtractor + ClaimScorer)")        try:            detector = BugDetector(                llm_model=LLM_MODEL,                embedding_model=EMBEDDING_MODEL,                use_llm_extraction=True,            )            scored: ScoredPlaybook = detector.analyze_playbook(                playbook_path,                api_records_path=entry.get("api_records_path"),                task_prompt=task_prompt,            )            # Build our per-interaction output (excluding overall_health_score)            interactions_out = []            for ix in scored.interactions:                interactions_out.append({                    "role": ix.role,                    "turn": ix.playbook_turn,                    "phase": ix.phase,                    "support_score__fm_2_2_2_3": ix.aggregate_support_score,                    "norm_score__fm_1_1_1_2": ix.aggregate_norm_score,                    "repetition_score__fm_1_3": ix.repetition_score,                    "plan_action_alignment__fm_2_6": ix.plan_action_alignment_score,                    "num_claims": ix.num_claims,                    "prompt_snippet": ix.prompt_snippet,                    "output_snippet": ix.output_snippet,                })            per_role = {}            for role in scored.roles:                role_ixs = [s for s in scored.interactions if s.role == role]                if not role_ixs:                    continue                per_role[role] = {                    "num_turns": len(role_ixs),                    "support_mean__fm_2_2_2_3": round(float(sum(s.aggregate_support_score for s in role_ixs) / len(role_ixs)), 4),                    "norm_mean__fm_1_1_1_2": round(float(sum(s.aggregate_norm_score for s in role_ixs) / len(role_ixs)), 4),                    "repetition_mean__fm_1_3": round(float(sum(s.repetition_score for s in role_ixs) / len(role_ixs)), 4),                    "plan_action_mean__fm_2_6": round(float(sum(s.plan_action_alignment_score for s in role_ixs) / len(role_ixs)), 4),                }            result = {                "source_log": entry["log_path"],                "mode": "two-step",                "task_prompt": task_prompt,                "num_roles": scored.num_roles,                "num_interactions": scored.num_interactions,                "roles": scored.roles,                "interactions": interactions_out,                "per_role_summary": per_role,                "aggregate_scores": {                    "support_score_mean": scored.support_score_mean,                    "support_score_min": scored.support_score_min,                    "norm_score_mean": scored.norm_score_mean,                    "norm_score_min": scored.norm_score_min,                    "repetition_score_mean": scored.repetition_score_mean,                    "repetition_score_max": scored.repetition_score_max,                    "plan_action_score_mean": scored.plan_action_score_mean,                    "plan_action_score_min": scored.plan_action_score_min,                },            }            all_results.append(result)            for role, summary in per_role.items():                print(f"  {role}: {summary['num_turns']} turns, "                      f"support={summary['support_mean__fm_2_2_2_3']:.3f}, "                      f"norm={summary['norm_mean__fm_1_1_1_2']:.3f}, "                      f"repetition={summary['repetition_mean__fm_1_3']:.3f}, "                      f"plan_action={summary['plan_action_mean__fm_2_6']:.3f}")        except Exception as e:            print(f"  ❌ Two-step scoring error: {e}")            import traceback            traceback.print_exc()    elif MODE == "llm-judge":        print(f"  Mode: llm-judge (GPT-4o evaluates 14 failure modes)")        try:            judge_results = run_llm_judge(playbook, task_prompt, _client, LLM_MODEL, TAXONOMY_DIR)            # Aggregate FM counts            fm_counts = defaultdict(int)            for jr in judge_results:                for fm, val in jr.get('failure_modes', {}).items():                    if val:                        fm_counts[fm] += 1            result = {                "source_log": entry["log_path"],                "mode": "llm-judge",                "task_prompt": task_prompt,                "num_interactions": len(judge_results),                "interactions": judge_results,                "failure_mode_frequency": {                    fm: {"name": fm_names.get(fm, fm), "count": fm_counts.get(fm, 0)}                    for fm in fm_names                },            }            all_results.append(result)        except Exception as e:            print(f"  ❌ LLM judge error: {e}")            import traceback            traceback.print_exc()print(f"\n{'='*60}")print(f"Scored {len(all_results)}/{len(playbook_cache)} logs")

## Step 4: Save Per-Log ResultsEach log's result is written as `{name}_scored.json` in `OUTPUT_DIR`.

In [ ]:
for result in all_results:    source_name = Path(result["source_log"]).stem    output_path = os.path.join(OUTPUT_DIR, f"{source_name}_scored.json")    with open(output_path, 'w', encoding='utf-8') as f:        json.dump(result, f, ensure_ascii=False, indent=2)    print(f"✓ {output_path}")print(f"\nSaved {len(all_results)} per-log JSON(s) to {os.path.abspath(OUTPUT_DIR)}")

## Step 5: Aggregate SummaryGenerates two files in `OUTPUT_DIR`:- `_summary.json` — overall statistics, per-FM frequencies, per-log summaries- `_summary.csv` — flat table: one row per (log, role, turn) for easy spreadsheet analysis

In [ ]:
# ---- Build aggregate summary JSON ----total_logs = len(all_results)total_interactions = sum(r.get('num_interactions', len(r.get('interactions', []))) for r in all_results)# Per-FM frequency (for llm-judge mode)fm_freq = defaultdict(lambda: {"count": 0, "name": ""})for r in all_results:    if r.get('mode') == 'llm-judge':        for fm, info in r.get('failure_mode_frequency', {}).items():            fm_freq[fm]["count"] += info.get("count", 0)            fm_freq[fm]["name"] = info.get("name", fm)# Score distributions (for two-step mode)two_step_results = [r for r in all_results if r.get('mode') == 'two-step']score_keys = ["support_score_mean", "norm_score_mean", "repetition_score_mean", "plan_action_score_mean"]score_dist = {}if two_step_results:    for key in score_keys:        vals = [r["aggregate_scores"][key] for r in two_step_results if r.get("aggregate_scores", {}).get(key) is not None]        if vals:            score_dist[key] = {                "min": round(min(vals), 4),                "max": round(max(vals), 4),                "mean": round(sum(vals) / len(vals), 4),            }summary = {    "mode": MODE,    "total_logs": total_logs,    "total_interactions": total_interactions,    "per_log": [        {            "name": Path(r["source_log"]).stem,            "num_interactions": r.get('num_interactions', len(r.get('interactions', []))),            "num_roles": r.get('num_roles', len(set(ix.get('role', '?') for ix in r.get('interactions', [])))),        }        for r in all_results    ],}if fm_freq:    summary["failure_mode_frequency"] = dict(fm_freq)if score_dist:    summary["score_distribution"] = score_distsummary_path = os.path.join(OUTPUT_DIR, "_summary.json")with open(summary_path, 'w', encoding='utf-8') as f:    json.dump(summary, f, ensure_ascii=False, indent=2)print(f"✓ {summary_path}")# ---- Build flat CSV ----csv_path = os.path.join(OUTPUT_DIR, "_summary.csv")csv_rows = []for r in all_results:    log_name = Path(r["source_log"]).stem    for ix in r.get("interactions", []):        row = {            "log_name": log_name,            "role": ix.get("role", ""),            "turn": ix.get("turn", 0),            "phase": ix.get("phase", ""),        }        # Add scores depending on mode        if r.get("mode") == "two-step":            row["support_score__fm_2_2_2_3"] = ix.get("support_score__fm_2_2_2_3", "")            row["norm_score__fm_1_1_1_2"] = ix.get("norm_score__fm_1_1_1_2", "")            row["repetition_score__fm_1_3"] = ix.get("repetition_score__fm_1_3", "")            row["plan_action_alignment__fm_2_6"] = ix.get("plan_action_alignment__fm_2_6", "")        elif r.get("mode") == "llm-judge":            row["task_completed"] = ix.get("task_completed", "")            row["summary"] = ix.get("summary", "")[:200]            for fm_code in fm_names:                row[f"fm_{fm_code.replace('.', '_')}"] = ix.get("failure_modes", {}).get(fm_code, "")        csv_rows.append(row)if csv_rows:    fieldnames = list(csv_rows[0].keys())    with open(csv_path, 'w', encoding='utf-8-sig', newline='') as f:        writer = csv.DictWriter(f, fieldnames=fieldnames)        writer.writeheader()        writer.writerows(csv_rows)    print(f"✓ {csv_path}  ({len(csv_rows)} rows × {len(fieldnames)} cols)")else:    print("⚠ No rows to write to CSV")print(f"\nAggregate files written to {os.path.abspath(OUTPUT_DIR)}")

## Step 6: Quick VisualizationFor two-step mode: bar charts of per-role average scores and per-log aggregate scores.For llm-judge mode: bar chart of failure mode frequencies.

In [ ]:
import matplotlib.pyplot as pltimport numpy as npif MODE == "two-step" and two_step_results:    fig, axes = plt.subplots(1, 2, figsize=(14, 5))    # ---- Per-log aggregate scores ----    log_names = [Path(r["source_log"]).stem[:40] for r in two_step_results]    x = np.arange(len(log_names))    width = 0.2    for j, (key, label) in enumerate([        ("support_score_mean", "Support (FM 2.2/2.3)"),        ("norm_score_mean", "Norm (FM 1.1/1.2)"),        ("repetition_score_mean", "Repetition (FM 1.3)"),        ("plan_action_score_mean", "Plan-Action (FM 2.6)"),    ]):        vals = [r["aggregate_scores"][key] for r in two_step_results]        axes[0].bar(x + j * width, vals, width, label=label)    axes[0].set_xticks(x + width * 1.5)    axes[0].set_xticklabels(log_names, rotation=45, ha='right', fontsize=8)    axes[0].set_ylabel("Score")    axes[0].set_title("Per-Log Aggregate Scores")    axes[0].legend(fontsize=7)    axes[0].set_ylim(0, 1)    axes[0].grid(axis='y', alpha=0.3)    # ---- Per-role averages across all logs ----    role_scores = defaultdict(lambda: defaultdict(list))    for r in two_step_results:        for role, summary_data in r.get("per_role_summary", {}).items():            role_scores[role]["support"].append(summary_data.get("support_mean__fm_2_2_2_3", 0))            role_scores[role]["norm"].append(summary_data.get("norm_mean__fm_1_1_1_2", 0))            role_scores[role]["repetition"].append(summary_data.get("repetition_mean__fm_1_3", 0))            role_scores[role]["plan_action"].append(summary_data.get("plan_action_mean__fm_2_6", 0))    roles_list = sorted(role_scores.keys())    x2 = np.arange(len(roles_list))    for j, (key, label) in enumerate([        ("support", "Support"), ("norm", "Norm"),        ("repetition", "Repetition (lower=better)"), ("plan_action", "Plan-Action"),    ]):        vals = [np.mean(role_scores[r][key]) for r in roles_list]        axes[1].bar(x2 + j * width, vals, width, label=label)    axes[1].set_xticks(x2 + width * 1.5)    axes[1].set_xticklabels(roles_list, rotation=45, ha='right', fontsize=8)    axes[1].set_ylabel("Mean Score")    axes[1].set_title("Per-Role Average Scores (across all logs)")    axes[1].legend(fontsize=7)    axes[1].set_ylim(0, 1)    axes[1].grid(axis='y', alpha=0.3)    plt.tight_layout()    plt.show()elif MODE == "llm-judge":    # Aggregate FM frequency across all logs    fm_total = defaultdict(int)    for r in all_results:        for fm, info in r.get("failure_mode_frequency", {}).items():            fm_total[fm] += info.get("count", 0)    if fm_total:        sorted_fm = sorted(fm_total.items(), key=lambda x: x[1], reverse=True)        codes = [f"{fm} {fm_names.get(fm, fm)}" for fm, _ in sorted_fm]        counts = [c for _, c in sorted_fm]        fig, ax = plt.subplots(figsize=(12, 5))        bars = ax.bar(range(len(codes)), counts)        ax.set_xticks(range(len(codes)))        ax.set_xticklabels(codes, rotation=45, ha='right', fontsize=8)        ax.set_ylabel("Detection Count")        ax.set_title(f"Failure Mode Frequency Across {total_logs} Log(s) — LLM Judge")        for bar, count in zip(bars, counts):            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,                    str(count), ha='center', fontsize=8)        ax.grid(axis='y', alpha=0.3)        plt.tight_layout()        plt.show()    else:        print("No failure modes detected.")else:    print("No data to visualize.")